<p align="center"><a href="https://www.plus2net.com/python/pandas-display-reset_option.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Pandas reset_option(): Restore Default Settings

Run cells in order. Requires Pandas and openpyxl. All student tables are embedded synthetic practice data. Excel practice uses temporary files. Try the exercise before the solution.

## Create a portable student table

The original tutorial displays 35 student rows after reading Excel with id as the index. This replacement uses explicitly synthetic records so the examples run without a local D: drive. It has four data columns after moving id into the index, correcting the original five-column description.

In [ ]:
import pandas as pd
students = pd.DataFrame({"id": range(1, 36),
    "name": [f"Student {i:02d}" for i in range(1, 36)],
    "class": ["Four"] * 35, "mark": [40 + i for i in range(35)],
    "sex": ["female" if i % 2 else "male" for i in range(35)]}).set_index("id")
print(students.head(3))
assert students.shape == (35, 4)

**Expected output**

```text
          name class  mark     sex
id                                
1   Student 01  Four    40    male
2   Student 02  Four    41  female
3   Student 03  Four    42    male
```

## Reset one option to its default

The original max_rows=6 example is retained. reset_option resets the registered default, not the value that existed before your change. A surrounding context restores your prior setting after this demonstration.

In [ ]:
saved = pd.get_option("display.max_rows")
with pd.option_context("display.max_rows", 6):
    print("Changed:", pd.get_option("display.max_rows"))
    returned = pd.reset_option("display.max_rows")
    print("Reset default:", pd.get_option("display.max_rows"))
    print(students.to_string(max_rows=35))
    assert returned is None and students.shape == (35, 4)
print("Prior setting restored:", pd.get_option("display.max_rows") == saved)
assert pd.get_option("display.max_rows") == saved

**Expected output**

```text
Changed: 6
Reset default: 60
          name class  mark     sex
id                                
1   Student 01  Four    40    male
2   Student 02  Four    41  female
3   Student 03  Four    42    male
4   Student 04  Four    43  female
5   Student 05  Four    44    male
6   Student 06  Four    45  female
7   Student 07  Four    46    male
8   Student 08  Four    47  female
9   Student 09  Four    48    male
10  Student 10  Four    49  female
11  Student 11  Four    50    male
12  Student 12  Four    51  female
13  Student 13  Four    52    male
14  Student 14  Four    53  female
15  Student 15  Four    54    male
16  Student 16  Four    55  female
17  Student 17  Four    56    male
18  Student 18  Four    57  female
19  Student 19  Four    58    male
20  Student 20  Four    59  female
21  Student 21  Four    60    male
22  Student 22  Four    61  female
23  Student 23  Four    62    male
24  Student 24  Four    63  female
25  Student 25  Four    64    male
26  Student 26  Four    65  female
27  Student 27  Four    66    male
28  Student 28  Four    67  female
29  Student 29  Four    68    male
30  Student 30  Four    69  female
31  Student 31  Four    70    male
32  Student 32  Four    71  female
33  Student 33  Four    72    male
34  Student 34  Four    73  female
35  Student 35  Four    74    male
Prior setting restored: True
```

## Default reset versus previous-value restoration

Use <a href="pandas-display-option_context.php">option_context()</a> for temporary changes. Resetting a default is different from returning to a customised prior value.

In [ ]:
before = pd.get_option("display.max_rows")
with pd.option_context("display.max_rows", 12):
    with pd.option_context("display.max_rows", 6):
        pd.reset_option("display.max_rows")
        print("Registered default:", pd.get_option("display.max_rows"))
    print("Previous custom value:", pd.get_option("display.max_rows"))
print("Outer restored:", pd.get_option("display.max_rows") == before)
assert pd.get_option("display.max_rows") == before

**Expected output**

```text
Registered default: 60
Previous custom value: 12
Outer restored: True
```

## Reset selected display options

A regular expression can reset several matching names. Anchor the pattern to a small known set when that is your intent. Full option names are clearer for single resets.

In [ ]:
rows_before = pd.get_option("display.max_rows")
cols_before = pd.get_option("display.max_columns")
with pd.option_context("display.max_rows", 6, "display.max_columns", 2):
    pd.reset_option(r"^display\.max_(rows|columns)$")
    print("Row default:", pd.get_option("display.max_rows"))
    print("Column default:", pd.get_option("display.max_columns"))
assert pd.get_option("display.max_rows") == rows_before
assert pd.get_option("display.max_columns") == cols_before

**Expected output**

```text
Row default: 60
Column default: 0
```

## Reset display options or all options deliberately

The original pd.reset_option("display") resets matching display settings, not every Pandas setting. pd.reset_option("all") resets all registered options. Both affect shared state, so this runnable example executes the display reset in an isolated Python process and leaves the all-option reset as an optional comment. Broad patterns may include deprecated options and emit warnings depending on version.

In [ ]:
import subprocess
import sys
worker = """import pandas as pd
pd.set_option('display.max_rows', 6)
pd.reset_option('display')
display_default = pd.get_option('display.max_rows')
assert display_default is not None
# Optional broad reset; may need plotting dependencies and emit warnings:
# pd.reset_option('all')
print('Display reset executed in isolated process')
"""
run = subprocess.run([sys.executable, "-c", worker], capture_output=True, text=True, check=True)
print(run.stdout.strip())
assert "reset executed" in run.stdout

**Expected output**

```text
Display reset executed in isolated process
```

## Retain the Excel input workflow

For your own file, read_excel(path, index_col="id") preserves the original import idea. Here an embedded sample creates a temporary workbook, reads it and cleans it automatically. Requires openpyxl. You can also use the original <a href="pandas-dataframe-exercise1.php">student Excel exercise</a>.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path
with TemporaryDirectory(dir=".") as folder:
    workbook = Path(folder) / "student.xlsx"
    students.reset_index().to_excel(workbook, index=False, engine="openpyxl")
    loaded = pd.read_excel(workbook, index_col="id")
    print("Imported shape:", loaded.shape)
    pd.testing.assert_frame_equal(loaded, students)
print("Temporary workbook removed:", not workbook.exists())

**Expected output**

```text
Imported shape: (35, 4)
Temporary workbook removed: True
```

## Common mistakes

Do not assume every session still uses a default max_rows of 60. Use <a href="pandas-display-describe_option.php">describe_option()</a> to inspect the installed version. A reset changes rendering, not the table contents. Partial patterns can match more settings as new options are added; use precise names. Prefer <a href="pandas-display-options-get-set.php">get_option()/set_option()</a> for inspection and intentional persistent changes. The original student workbook can be used through the <a href="pandas-dataframe-exercise1.php">Excel exercise</a>.

## Exercise: reset a column limit

Temporarily change display.max_columns to two, reset only that option, and verify that leaving the context restores your earlier value.

## Exercise solution

Capture the previous setting and constrain the demonstration with a context.

In [ ]:
previous = pd.get_option("display.max_columns")
with pd.option_context("display.max_columns", 2):
    pd.reset_option("display.max_columns")
    print("Reset columns:", pd.get_option("display.max_columns"))
print("Prior value restored:", pd.get_option("display.max_columns") == previous)
assert pd.get_option("display.max_columns") == previous

**Expected output**

```text
Reset columns: 0
Prior value restored: True
```